In [1]:
import tiktoken

class GPT_Tokenizer:
   def __init__(self):
      self.enc = tiktoken.get_encoding("gpt2")

   def encode(self, text):
       self.encoding = self.enc.encode(text)
       return self.encoding

   def decode(self, token):
       self.decoding = self.enc.decode(token)
       return self.decoding

GPT-2 tokenizer implementation using OpenAI's tiktoken library.
Provides encoding and decoding functionality compatible with OpenAI's GPT models.

In [2]:
with open("iLoveMerge.txt", "r", encoding="utf-8") as file:
   raw_data = file.read()

Read the text file containing training data for tokenization and model training

In [3]:
text_data = raw_data[:100]
print(text_data)

**Welcome To The World of Free Plain Vanilla Electronic Texts**

**Etexts Readable By Both Humans an


### For demonstration, using the first 100 characters which will be encoded later

In [4]:
tokenizer = GPT_Tokenizer()
encode_text = tokenizer.encode(text_data)
print(encode_text)

[1174, 14618, 1675, 383, 2159, 286, 3232, 28847, 33897, 19508, 8255, 82, 1174, 198, 198, 1174, 36, 5239, 82, 4149, 540, 2750, 5747, 27411, 281]


### Tokenization complete: words converted to tokens using tiktoken

In [5]:
decode_text = tokenizer.decode(encode_text)
print(decode_text)

**Welcome To The World of Free Plain Vanilla Electronic Texts**

**Etexts Readable By Both Humans an


### Decoding reproduces the exact original text that was encoded

## Prepares the dataset for LLM training using PyTorch's DataLoader.

### This structures the data into batches, enabling efficient GPU utilization
### and shuffling. For next-token prediction (causal LM), the target sequence
### is the input sequence shifted right by one position.

In [6]:
import torch
from torch.utils.data import Dataset, DataLoader

class Implement_Dataset(Dataset):
    def __init__(self, text, max_length, stride):
        self.input_id = []
        self.target_id = []

        tokenizer = GPT_Tokenizer()
        token_id = tokenizer.encode(text)

        # range should use step=stride, not a second argument
        for i in range(0, len(token_id) - max_length, stride):
            input_seq = token_id[i : i + max_length]
            target_seq = token_id[i + 1 : i + max_length + 1]

            self.input_id.append(input_seq)
            self.target_id.append(target_seq)

    def __len__(self):
        return len(self.input_id)

    def __getitem__(self, idx):
        return torch.tensor(self.input_id[idx]), torch.tensor(self.target_id[idx])


This code implements a custom dataset class for our LLM training. Next, we’ll create a DataLoader to fetch the data in batches and iterate over it during training.

In [7]:
def Implement_DataLoader(txt,batch_size=4, max_length=256, stride=128, shuffle=True, drop_last=True,num_workers=0):
      dataset = Implement_Dataset(txt, max_length, stride)
      dataloader = DataLoader(dataset,
                              batch_size= batch_size ,
                              shuffle= shuffle,
                              drop_last= drop_last ,
                              num_workers=num_workers)
      return dataloader

In [8]:
dataloader = Implement_DataLoader(text_data, batch_size=4, max_length=8, stride=4)

data_iter = iter(dataloader)
# first_batch = next(data_iter)
# print(first_batch)
inputs, targets = next(data_iter)
print("Inputs:\n", inputs)
print("\nTargets:\n", targets)

Inputs:
 tensor([[33897, 19508,  8255,    82,  1174,   198,   198,  1174],
        [   36,  5239,    82,  4149,   540,  2750,  5747, 27411],
        [ 1174, 14618,  1675,   383,  2159,   286,  3232, 28847],
        [ 1174,   198,   198,  1174,    36,  5239,    82,  4149]])

Targets:
 tensor([[19508,  8255,    82,  1174,   198,   198,  1174,    36],
        [ 5239,    82,  4149,   540,  2750,  5747, 27411,   281],
        [14618,  1675,   383,  2159,   286,  3232, 28847, 33897],
        [  198,   198,  1174,    36,  5239,    82,  4149,   540]])


### Implementing Embeddings
### Combines token embeddings with positional encodings to give the model
### information about both word identity and sequence position.

In [9]:
import torch.nn as nn

class Embedding(nn.Module):
      def __init__(self, emb_dim, vocab_size, max_length):
        self.token_emb = nn.Embedding(vocab_size, emb_dim)
        self.pos_emb = nn.Embedding(max_length, emb_dim)

      def forward(self, x):
          positions = torch.arange(0, x.size(1), device=x.device).unsqueeze(0)
          return self.token_emb(x) + self.pos_emb(positions)

### The core of GPT is the attention mechanism inside the Transformer. In the code below, I implement it step by step: first by initializing the query, key, and value projection weights, then by adding causal attention to prevent tokens from looking ahead. I also include dropout to improve generalization and reduce overfitting.

In [10]:

class MultiHeadAttention(nn.Module):
    def __init__(self, d_in, d_out, context_length, dropout, num_heads, qkv_bias=False):
        super().__init__()

        # Ensure even split across heads
        assert d_out % num_heads == 0, "d_out must be divisible by num_heads"

        self.d_out = d_out
        self.num_heads = num_heads
        self.head_dim = d_out // num_heads

        # Linear projections for Q, K, V
        self.W_query = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_key   = nn.Linear(d_in, d_out, bias=qkv_bias)
        self.W_value = nn.Linear(d_in, d_out, bias=qkv_bias)

        # Final output projection
        self.out_proj = nn.Linear(d_out, d_out)

        self.dropout = nn.Dropout(dropout)

        # Causal mask (upper triangular = 1 -> masked)
        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length), diagonal=1)
        )

    def forward(self, x):
        b, seq_len, _ = x.shape

        # Compute Q, K, V
        Q = self.W_query(x)
        K = self.W_key(x)
        V = self.W_value(x)

        # Reshape for multi-head: (b, seq_len, num_heads, head_dim)
        Q = Q.view(b, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        K = K.view(b, seq_len, self.num_heads, self.head_dim).transpose(1, 2)
        V = V.view(b, seq_len, self.num_heads, self.head_dim).transpose(1, 2)

        # Attention scores: (b, num_heads, seq_len, seq_len)
        scores = Q @ K.transpose(2, 3)

        # Causal masking
        mask = self.mask.bool()[:seq_len, :seq_len]
        scores = scores.masked_fill(mask, float("-inf"))

        # Softmax over last dimension
        attn = torch.softmax(scores / (self.head_dim ** 0.5), dim=-1)
        attn = self.dropout(attn)

        # Weighted sum of values
        context = attn @ V   # (b, num_heads, seq_len, head_dim)

        # Merge heads back: (b, seq_len, d_out)
        context = context.transpose(1, 2).contiguous().view(b, seq_len, self.d_out)

        return self.out_proj(context)


In [11]:
import torch
torch.manual_seed(123)


inputs = torch.tensor(
    [[0.43, 0.15, 0.89, 0.55, 0.87, 0.66],  # Row 1
     [0.57, 0.85, 0.64, 0.22, 0.58, 0.33],  # Row 2
     [0.77, 0.25, 0.10, 0.05, 0.80, 0.55]]  # Row 3
)

batch = torch.stack((inputs, inputs), dim=0)
print(batch.shape)

batch_size, context_length, d_in = batch.shape
d_out = 6
mha = MultiHeadAttention(d_in, d_out, context_length, 0.0, num_heads=2)
context_vecs = mha(batch)
print(context_vecs)
print("context_vecs.shape:", context_vecs.shape)

torch.Size([2, 3, 6])
tensor([[[ 0.1569, -0.0873,  0.0210,  0.0215, -0.3243, -0.2518],
         [ 0.1117, -0.0547,  0.0406, -0.0213, -0.3251, -0.2993],
         [ 0.1196, -0.0491,  0.0318, -0.0635, -0.2788, -0.2578]],

        [[ 0.1569, -0.0873,  0.0210,  0.0215, -0.3243, -0.2518],
         [ 0.1117, -0.0547,  0.0406, -0.0213, -0.3251, -0.2993],
         [ 0.1196, -0.0491,  0.0318, -0.0635, -0.2788, -0.2578]]],
       grad_fn=<ViewBackward0>)
context_vecs.shape: torch.Size([2, 3, 6])


In [12]:
text_demo = "Hello Every one"
tokenizer = GPT_Tokenizer()
tokenized = tokenizer.encode(text_demo)
print(tokenized)

[15496, 3887, 530]


In [13]:
# GPT_CONFIG_124M = {
#     "vocab_size": 50257,
#     "context_length": 1024,
#     "emb_dim": 768,
#     "n_heads": 12,
#     "n_layers": 12,
#     "drop_rate": 0.1,
#     "qkv_bias": False
# }
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}

Now let's implement layer normalization

In [14]:
import torch.nn as nn

class LayerNorm(nn.Module):
    def __init__(self, emb_dim):
        super().__init__()
        self.eps = 1e-5
        self.scale = nn.Parameter(torch.ones(emb_dim))
        self.shift = nn.Parameter(torch.zeros(emb_dim))

    def forward(self, x):
        mean = x.mean(dim=-1, keepdim=True)
        var = x.var(dim=-1, keepdim=True, unbiased=False)
        norm_x = (x - mean) / torch.sqrt(var + self.eps)
        return self.scale * norm_x + self.shift


In [15]:
import torch
batch_example = torch.randn(2, 5) #A

In [16]:
print(batch_example)

tensor([[-0.6669,  0.5074, -1.1026, -0.3533, -1.7799],
        [ 0.6474,  0.5460,  0.8050, -1.3467, -0.6418]])


In [17]:
import torch.nn as nn

ln = LayerNorm(emb_dim=5)
out_ln = ln(batch_example)
mean = out_ln.mean(dim=-1, keepdim=True)
var = out_ln.var(dim=-1, unbiased=False, keepdim=True)

print("Mean:\n", mean)
print("Variance:\n", var)

Mean:
 tensor([[-5.9605e-09],
        [-4.7684e-08]], grad_fn=<MeanBackward1>)
Variance:
 tensor([[1.0000],
        [1.0000]], grad_fn=<VarBackward0>)


In [18]:

class GELU(nn.Module):
    def __init__(self):
        super().__init__()

    def forward(self, x):
        return 0.5 * x * (1 + torch.tanh(
            torch.sqrt(torch.tensor(2.0 / torch.pi)) *
            (x + 0.044715 * torch.pow(x, 3))
        ))

In [19]:
class FeedForward(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.layers = nn.Sequential(
            nn.Linear(cfg["emb_dim"], 4 * cfg["emb_dim"]), ## Expansion
            GELU(), ## Activation
            nn.Linear(4 * cfg["emb_dim"], cfg["emb_dim"]), ## Contraction
        )

    def forward(self, x):
        return self.layers(x)

In [20]:
class TransformerBlock(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.att = MultiHeadAttention(
            d_in=cfg["emb_dim"],
            d_out=cfg["emb_dim"],
            context_length=cfg["context_length"],
            num_heads=cfg["n_heads"],
            dropout=cfg["drop_rate"],
            qkv_bias=cfg["qkv_bias"])
        self.ff = FeedForward(cfg)
        self.norm1 = LayerNorm(cfg["emb_dim"])
        self.norm2 = LayerNorm(cfg["emb_dim"])
        self.drop_shortcut = nn.Dropout(cfg["drop_rate"])

    def forward(self, x):
        # Shortcut connection for attention block
        shortcut = x
        x = self.norm1(x)
        x = self.att(x)  # Shape [batch_size, num_tokens, emb_size]
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        # Shortcut connection for feed forward block
        shortcut = x
        x = self.norm2(x)
        x = self.ff(x)
        # 2*4*768
        x = self.drop_shortcut(x)
        x = x + shortcut  # Add the original input back

        return x
        # 2*4*768

In [21]:
class GPTModel(nn.Module):
    def __init__(self, cfg):
        super().__init__()
        self.tok_emb = nn.Embedding(cfg["vocab_size"], cfg["emb_dim"])
        self.pos_emb = nn.Embedding(cfg["context_length"], cfg["emb_dim"])
        self.drop_emb = nn.Dropout(cfg["drop_rate"])

        self.trf_blocks = nn.Sequential(
            *[TransformerBlock(cfg) for _ in range(cfg["n_layers"])])

        self.final_norm = LayerNorm(cfg["emb_dim"])
        self.out_head = nn.Linear(
            cfg["emb_dim"], cfg["vocab_size"], bias=False
        )

    def forward(self, in_idx):
        batch_size, seq_len = in_idx.shape
        tok_embeds = self.tok_emb(in_idx)
        pos_embeds = self.pos_emb(torch.arange(seq_len, device=in_idx.device))
        x = tok_embeds + pos_embeds
        x = self.drop_emb(x)
        x = self.trf_blocks(x)
        x = self.final_norm(x)
        logits = self.out_head(x)
        return logits

In [22]:
import tiktoken
tokenizer = GPT_Tokenizer()
batch = []
txt1 = "Your hard work leads to"
txt2 = "hard work is a key"
batch.append(torch.tensor(tokenizer.encode(txt1)))
batch.append(torch.tensor(tokenizer.encode(txt2)))
batch = torch.stack(batch, dim=0)
print(batch)

tensor([[ 7120,  1327,   670,  5983,   284],
        [10424,   670,   318,   257,  1994]])


In [23]:
torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
out = model(batch)
print("Input batch:\n", batch)
print("\nOutput shape:", out.shape)
print(out)

Input batch:
 tensor([[ 7120,  1327,   670,  5983,   284],
        [10424,   670,   318,   257,  1994]])

Output shape: torch.Size([2, 5, 50257])
tensor([[[-0.7361, -0.6564, -0.6750,  ..., -0.0215,  0.4834, -0.4088],
         [-0.9163,  0.0315, -0.3796,  ..., -0.2452,  0.1635,  0.0726],
         [ 0.6389,  0.1410, -0.2927,  ..., -0.8692,  1.0700, -0.0491],
         [ 0.5258, -0.0775, -0.3521,  ...,  0.0209,  0.7328,  0.0315],
         [ 0.8585, -0.1910,  0.1566,  ..., -0.3078,  0.9366,  0.5343]],

        [[-0.3328,  0.0982, -0.3573,  ..., -0.4725,  0.1461,  0.4045],
         [-0.8475, -0.4993, -0.8165,  ..., -0.4942, -0.2165,  0.3533],
         [ 0.4581, -0.7552,  0.0493,  ..., -0.1302,  0.2099, -0.5708],
         [-0.5362, -0.2466,  0.0231,  ..., -0.4570,  0.4583, -0.5283],
         [-1.0173, -0.2866,  0.5633,  ..., -0.0995,  0.2351,  0.4828]]],
       grad_fn=<UnsafeViewBackward0>)


In [24]:
def generate_text_simple(model, idx, max_new_tokens, context_size):
    # idx is (batch, n_tokens) array of indices in the current context
    for _ in range(max_new_tokens):

        # then only the last context_size tokens are used as context 10 token 5
        idx_cond = idx[:, -context_size:]

        # Get the predictions
        with torch.no_grad():
            logits = model(idx_cond)

        # Focus only on the last time step
        logits = logits[:, -1, :]
        # Apply softmax to get probabilities
        probas = torch.softmax(logits, dim=-1)  # (batch, vocab_size)

        # Get the idx of the vocab entry with the highest probability value
        idx_next = torch.argmax(probas, dim=-1, keepdim=True)  # (batch, 1)
        # Append sampled index to the running sequence
        idx = torch.cat((idx, idx_next), dim=1)  # (batch, n_tokens+1)

    return idx

In [25]:
start_context = "Hello,I'm a developer"
encoded = tokenizer.encode(start_context)
print("encoded:", encoded)
encoded_tensor = torch.tensor(encoded).unsqueeze(0) #A
print("encoded_tensor.shape:", encoded_tensor.shape)

encoded: [15496, 11, 40, 1101, 257, 8517]
encoded_tensor.shape: torch.Size([1, 6])


In [26]:
model.eval() #A
#model = GPTModel(GPT_CONFIG_124M)
out = generate_text_simple(
model=model,
idx=encoded_tensor,
max_new_tokens=10,
context_size=GPT_CONFIG_124M["context_length"]
)
print("Output:", out)
print("Output length:", len(out[0]))

Output: tensor([[15496,    11,    40,  1101,   257,  8517, 29378, 43530, 19334, 10983,
          6842, 39881,  2091, 33703, 26985, 16699]])
Output length: 16


In [27]:
decoded_text = tokenizer.decode(out.squeeze(0).tolist())
print(decoded_text)

Hello,I'm a developer shareholder neoconsdraw Hat bear Assy33 Huck Von Almost


calcuate cross-entropy loss

In [28]:
with open("iLoveMerge.txt", "r") as f :
   content = f.read()

In [29]:
print(content[:100])

**Welcome To The World of Free Plain Vanilla Electronic Texts**

**Etexts Readable By Both Humans an


In [30]:
print(len(content))

94150


In [31]:
tokenizer = GPT_Tokenizer()
tokenized = tokenizer.encode(content)
print(len(tokenized))

23848


In [32]:
GPT_CONFIG_124M = {
    "vocab_size": 50257,   # Vocabulary size
    "context_length": 256, # Shortened context length (orig: 1024)
    "emb_dim": 768,        # Embedding dimension
    "n_heads": 12,         # Number of attention heads
    "n_layers": 12,        # Number of layers
    "drop_rate": 0.1,      # Dropout rate
    "qkv_bias": False      # Query-key-value bias
}

In [33]:
# Train/validation ratio
train_ratio = 0.90
split_idx = int(train_ratio * len(content))
train_data = content[:split_idx]
val_data = content[split_idx:]


torch.manual_seed(123)

train_loader = Implement_DataLoader(
    train_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=True,
    shuffle=True,
    num_workers=0
)

val_loader = Implement_DataLoader(
    val_data,
    batch_size=2,
    max_length=GPT_CONFIG_124M["context_length"],
    stride=GPT_CONFIG_124M["context_length"],
    drop_last=False,
    shuffle=False,
    num_workers=0
)

In [34]:
print("Train loader:")
for x, y in train_loader:
    print(x.shape, y.shape)

print("\nValidation loader:")
for x, y in val_loader:
    print(x.shape, y.shape)

print(len(train_loader))
print(len(val_loader))

Train loader:
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256]) torch.Size([2, 256])
torch.Size([2, 256])

In [35]:
train_tokens = 0
for input_batch, target_batch in train_loader:
    train_tokens += input_batch.numel()

val_tokens = 0
for input_batch, target_batch in val_loader:
    val_tokens += input_batch.numel()

print("Training tokens:", train_tokens)
print("Validation tokens:", val_tokens)
print("All tokens:", train_tokens + val_tokens)

Training tokens: 20992
Validation tokens: 2304
All tokens: 23296


In [36]:
# def calc_batch_loss(input_txt, target_txt, model, device):
#   input_txt , target_txt = input_txt.to(device), target_txt.to(device)
#   logits = model(input_txt)
#   loss = torch.nn.functional.cross_entropy(logits.flatten(0, 1), target_batch.flatten())
#   return loss
def calc_batch_loss(input_txt, target_txt, model, device):
    input_txt, target_txt = input_txt.to(device), target_txt.to(device)

    logits = model(input_txt)                    # (B, T, vocab)

    # Flatten both consistently
    logits = logits.flatten(0, 1)                # (B*T, vocab)
    target_txt = target_txt.flatten()            # (B*T)

    loss = torch.nn.functional.cross_entropy(logits, target_txt)
    return loss


In [37]:
def calc_loss_loader(data_loader, model, device, num_batches=None):
    total_loss = 0.
    if len(data_loader) == 0:
        return float("nan")
    elif num_batches is None:
        num_batches = len(data_loader)
    else:
        num_batches = min(num_batches, len(data_loader))
    for i, (input_batch, target_batch) in enumerate(data_loader):
        if i < num_batches:
            loss = calc_batch_loss(input_batch, target_batch, model, device)
            total_loss += loss.item()
        else:
            break
    return total_loss / num_batches

In [38]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Note:
# Uncommenting the following lines will allow the code to run on Apple Silicon chips, if applicable,
# which is approximately 2x faster than on an Apple CPU (as measured on an M3 MacBook Air).
# However, the resulting loss values may be slightly different.

#if torch.cuda.is_available():
#    device = torch.device("cuda")
#elif torch.backends.mps.is_available():
#    device = torch.device("mps")
#else:
#    device = torch.device("cpu")
#
# print(f"Using {device} device.")


model.to(device) # no assignment model = model.to(device) necessary for nn.Module classes


torch.manual_seed(123) # For reproducibility due to the shuffling in the data loader

with torch.no_grad(): # Disable gradient tracking for efficiency because we are not training, yet
    train_loss = calc_loss_loader(train_loader, model, device)
    val_loss = calc_loss_loader(val_loader, model, device)

print("Training loss:", train_loss)
print("Validation loss:", val_loss)

Training loss: 10.990123353353361
Validation loss: 11.004333686828613


### Traning Loop

In [39]:
def evaluate_model(model, train_loader, val_loader, device, eval_iter):
    model.eval()
    with torch.no_grad():
        train_loss = calc_loss_loader(train_loader, model, device, num_batches=eval_iter)
        val_loss = calc_loss_loader(val_loader, model, device, num_batches=eval_iter)
    model.train()
    return train_loss, val_loss

In [40]:
def generate_and_print_sample(model, tokenizer, device, start_context):
    model.eval()

    context_size = model.pos_emb.weight.shape[0]

    encoded = torch.tensor(
        tokenizer.encode(start_context),
        dtype=torch.long
    ).unsqueeze(0).to(device)

    with torch.no_grad():
        token_ids = generate_text_simple(
            model=model,
            idx=encoded,
            max_new_tokens=50,
            context_size=context_size
        )

    decoded_text = tokenizer.decode(token_ids[0].tolist())
    print(decoded_text.replace("\n", " "))

    model.train()


In [41]:
def train_model_simple(model, train_loader, val_loader, optimizer, device,
                       num_epochs, eval_freq, eval_iter, start_context, tokenizer):

    # Tracking history
    train_losses = []
    val_losses = []
    token_progress = []

    tokens_processed = 0
    step = 0

    for epoch in range(1, num_epochs + 1):
        model.train()   # Enable training mode

        for inputs, targets in train_loader:
            # Reset previous gradients
            optimizer.zero_grad()

            # Forward + Loss
            loss = calc_batch_loss(inputs, targets, model, device)

            # Backpropagation
            loss.backward()

            # Parameter update
            optimizer.step()

            # Tracking progress
            tokens_processed += inputs.numel()
            step += 1

            # Run evaluation periodically
            if step % eval_freq == 0:
                train_loss, val_loss = evaluate_model(
                    model=model,
                    train_loader=train_loader,
                    val_loader=val_loader,
                    device=device,
                    eval_iter=eval_iter
                )

                train_losses.append(train_loss)
                val_losses.append(val_loss)
                token_progress.append(tokens_processed)

                print(
                    f"Epoch {epoch} | Step {step:06d} | "
                    f"Train Loss: {train_loss:.3f} | Val Loss: {val_loss:.3f}"
                )

        # Generate example output after each epoch
        generate_and_print_sample(model, tokenizer, device, start_context)

    return train_losses, val_losses, token_progress


In [ ]:
import time
start_time = time.time()

torch.manual_seed(123)
model = GPTModel(GPT_CONFIG_124M)
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=0.0004, weight_decay=0.1)

num_epochs = 10
train_losses, val_losses, tokens_seen = train_model_simple(
    model, train_loader, val_loader, optimizer, device,
    num_epochs=num_epochs, eval_freq=5, eval_iter=5,
    start_context="It is never too late to", tokenizer=tokenizer
)

end_time = time.time()
execution_time_minutes = (end_time - start_time) / 60
print(f"Training completed in {execution_time_minutes:.2f} minutes.")

Epoch 1 | Step 000005 | Train Loss: 8.296 | Val Loss: 8.399
Epoch 1 | Step 000010 | Train Loss: 6.987 | Val Loss: 7.398
Epoch 1 | Step 000015 | Train Loss: 6.833 | Val Loss: 6.959
Epoch 1 | Step 000020 | Train Loss: 5.875 | Val Loss: 6.855
Epoch 1 | Step 000025 | Train Loss: 6.382 | Val Loss: 6.794
Epoch 1 | Step 000030 | Train Loss: 6.044 | Val Loss: 6.733
Epoch 1 | Step 000035 | Train Loss: 5.460 | Val Loss: 6.696
Epoch 1 | Step 000040 | Train Loss: 5.558 | Val Loss: 6.637
It is never too late to                                                  
Epoch 2 | Step 000045 | Train Loss: 5.632 | Val Loss: 6.643
Epoch 2 | Step 000050 | Train Loss: 5.611 | Val Loss: 6.631
Epoch 2 | Step 000055 | Train Loss: 5.144 | Val Loss: 6.698
Epoch 2 | Step 000060 | Train Loss: 5.158 | Val Loss: 6.832
Epoch 2 | Step 000065 | Train Loss: 5.078 | Val Loss: 6.553
Epoch 2 | Step 000070 | Train Loss: 5.036 | Val Loss: 6.420
Epoch 2 | Step 000075 | Train Loss: 4.902 | Val Loss: 6.388
Epoch 2 | Step 000080 | Tr